# SmolVLM-500M LoRA Finetune (Kaggle 2xT4, 10k train)

**Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (500M, public, no token) — LoRA r32 fp16 (T4 has no BF16), no quantization needed at this size.

**Data:** `/kaggle/input/competitions/astroclimb/train.csv` (OLD 10k, base64 obj_1/obj_2) → `val 1800 balanced` + `train 8200`.

**Hardware:** Kaggle `GPU T4 x2`, `device_map=auto`, `BATCH 1x16 eff 16`, `MAX_SEQ_LEN 2048`.

**Loss:** answer-only supervision (prompt tokens masked to -100, pads masked, label never truncated).

**Saves:** local ckpts every 200 + intermediate pushes to HF `checkpoints/` + final `adapter_best` + `eval/` metrics.


In [1]:
# --- 0. Config (repo auto-created from your login, just add HF_TOKEN) ---
MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
HF_REPO_NAME = 'smolvlm-500m-lora-astroclimb'
HF_REPO_ID = None  # auto-set to username/repo after HF login, code creates it
HF_PRIVATE = True
TRAIN_CSV = '/kaggle/input/competitions/astroclimb/train.csv'
VAL_N_PER_CLASS = 450
EPOCHS = 10
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
LORA_R = 32
LORA_ALPHA = 64
MAX_SEQ_LEN = 2048
CAP_CHARS = 1000
SAVE_STEPS = 200
EVAL_STEPS = 200
MAX_STEPS = 5120
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001
MAX_NEW_TOKENS = 32
TIME_BUDGET = 8*3600
LABEL_COLS = ['same_figure','same_paper','related_papers','unrelated_papers']
CKPT_DIR = '/kaggle/working/smolvlm_500m_lora_out'
FINAL_DIR = '/kaggle/working/smolvlm_500m_lora_final'
EVAL_DIR = '/kaggle/working/smolvlm_500m_eval'
import os, random, numpy as np
random.seed(42); np.random.seed(42)
print('MODEL=' + MODEL_ID + ' lora r=' + str(LORA_R))
print('HF repo auto: username/' + HF_REPO_NAME + ' (id set after login)')
print('TRAIN_CSV=' + TRAIN_CSV)


MODEL=HuggingFaceTB/SmolVLM-500M-Instruct lora r=32
HF repo auto: username/smolvlm-500m-lora-astroclimb (id set after login)
TRAIN_CSV=/kaggle/input/competitions/astroclimb/train.csv


In [2]:
# --- 1. Setup (Kaggle T4 x2: torch>=2.5 + transformers>=4.48, auto-upgrade) ---
import os, json, time, re, gc, base64, sys, subprocess
from pathlib import Path
from io import BytesIO
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import importlib.metadata as _im
from packaging import version as _pv
try:
    _torch_v = _im.version('torch')
except Exception: _torch_v = '0.0.0'
print('installed torch ' + _torch_v)
if _pv.parse(_torch_v) < _pv.parse('2.5.0'):
    print('Upgrading torch to 2.5.1+cu121 (transformers needs torch>=2.5)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch==2.5.1', 'torchaudio==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121'])
    print('torch cu121 installed')
for m in list(sys.modules.keys()):
    if m.split('.')[0] in ('torch', 'torchvision', 'torchaudio', 'transformers', 'peft', 'accelerate', 'typing_extensions'): del sys.modules[m]
import importlib as _il; _il.invalidate_caches()
import torch
if not torch.cuda.is_available():
    print('cu121 has no CUDA here - falling back to cu118 build')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch==2.5.1', 'torchaudio==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu118'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('torch', 'torchvision', 'torchaudio'): del sys.modules[m]
    _il.invalidate_caches()
    import torch
print('torch ' + torch.__version__ + ' cuda ' + str(torch.cuda.is_available()))
assert torch.cuda.is_available(), 'Enable GPU: Settings -> Accelerator -> GPU T4 x2'
assert torch.cuda.device_count() >= 2, 'Need 2xT4'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
_need = _pv.parse(_im.version('transformers')) < _pv.parse('4.48.0')
print('transformers ' + _im.version('transformers') + ' need>=4.48: ' + str(_need))
if _need:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.48', 'accelerate', 'peft'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('transformers', 'peft', 'accelerate', 'typing_extensions'): del sys.modules[m]
    _il.invalidate_caches()
import transformers; print('transformers ' + transformers.__version__)
import peft; print('peft ' + peft.__version__)
try:
    import torchao as _ta
    print('stale torchao ' + _ta.__version__ + ' present - removing (breaks peft, unused here)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] == 'torchao': del sys.modules[m]
    print('torchao removed')
except ImportError:
    print('no torchao present (fine)')
def _datasets_usable():
    try:
        import datasets as _d
        return _pv.parse(_d.__version__) >= _pv.parse('3.0.0')
    except Exception:
        return False
if not _datasets_usable():
    print('Repairing datasets/pyarrow install...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--force-reinstall', '--no-deps', 'datasets', 'pyarrow'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('datasets', 'pyarrow', 'dill', 'multiprocess', 'fsspec', 'huggingface_hub'): del sys.modules[m]
    _il.invalidate_caches()
if not _datasets_usable():
    print('datasets still broken - removing (Trainer works without it)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'datasets'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] == 'datasets': del sys.modules[m]
    print('datasets removed')
else:
    import datasets; print('datasets ' + datasets.__version__)
try: import seaborn as sns
except:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn']); import seaborn as sns


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


installed torch 2.0.0
Upgrading torch to 2.5.1+cu121 (transformers needs torch>=2.5)...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
cuml 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.3 which is incompatible.
cudf 23.8.0 requires protobuf<5,>=4.21, but you have protobuf 3.20.3 which is incompatible.
cuml 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
cuml 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, b

torch cu121 installed
torch 2.5.1+cu121 cuda True
['Tesla T4', 'Tesla T4']
transformers 4.36.0 need>=4.48: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
dask-cuda 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
dask-cuda 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cuda 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.3 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but 

transformers 5.16.1
peft 0.20.0
no torchao present (fine)
Repairing datasets/pyarrow install...
datasets still broken - removing (Trainer works without it)...
Found existing installation: datasets 5.0.1
Uninstalling datasets-5.0.1:
  Successfully uninstalled datasets-5.0.1
datasets removed


In [3]:
# --- 1b. HF login (optional — only needed for pushing checkpoints) ---
from huggingface_hub import login
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN from Kaggle Secrets')
    except Exception as e: print('No HF_TOKEN: ' + str(e))
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF login OK')
    try:
        from huggingface_hub import HfApi as _HfApi
        HF_REPO_ID = _HfApi().whoami()['name'] + '/' + HF_REPO_NAME
        _HfApi().create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
        print('Repo ready -> ' + HF_REPO_ID)
    except Exception as _e:
        HF_REPO_ID = None
        print('Repo setup failed, local-only: ' + str(_e)[:200])
else:
    print('No token — local saves only (/kaggle/working)')


HF_TOKEN from Kaggle Secrets
HF login OK
Repo ready -> ShivRamSaud/smolvlm-500m-lora-astroclimb


In [4]:
# --- 2. Data: Kaggle train.csv if attached, else HF dataset download ---
from huggingface_hub import hf_hub_download
HF_TRAIN_DATASET = 'ShivRamSaud/astroclimb_train'
if Path(TRAIN_CSV).exists():
    _csv = TRAIN_CSV
    print('Using Kaggle competition file')
else:
    print('Kaggle input missing - downloading from HF ' + HF_TRAIN_DATASET)
    _csv = hf_hub_download(repo_id=HF_TRAIN_DATASET, filename='train.csv', repo_type='dataset')
    print('Downloaded ' + str(_csv))
train = pd.read_csv(_csv)
print(train.shape)
print(train.columns.tolist())
train['label'] = train[LABEL_COLS].idxmax(axis=1)
train['label_id'] = train['label'].map({c:i for i,c in enumerate(LABEL_COLS)})
print(train['label'].value_counts())
def is_image_str(s):
    if not isinstance(s, str) or len(s) < 200: return False
    return s.strip().startswith('iVBORw0KGgo')
def convert_str_to_PIL(img_str):
    return Image.open(BytesIO(base64.b64decode(img_str))).convert('RGB')
train['obj_1_is_img'] = train['obj_1'].apply(is_image_str)
train['obj_2_is_img'] = train['obj_2'].apply(is_image_str)
train['pair_type'] = train.apply(lambda r: ('IMG' if r['obj_1_is_img'] else 'TXT') + '-' + ('IMG' if r['obj_2_is_img'] else 'TXT'), axis=1)
print(train['pair_type'].value_counts())


Kaggle input missing - downloading from HF ShivRamSaud/astroclimb_train


train.csv: reconstructing file:   0%|          |  0.00B / 10.1GB            

train.csv: downloading bytes:           |  0.00B            

Downloaded /root/.cache/huggingface/hub/datasets--ShivRamSaud--astroclimb_train/snapshots/f239cbb4a2ce525ec5f38caf81f00caa2eebb1bd/train.csv
(10000, 7)
['id', 'same_figure', 'same_paper', 'related_papers', 'unrelated_papers', 'obj_1', 'obj_2']
label
same_paper          3000
related_papers      3000
unrelated_papers    3000
same_figure         1000
Name: count, dtype: int64
pair_type
TXT-IMG    4000
IMG-IMG    3000
TXT-TXT    3000
Name: count, dtype: int64


In [5]:
# --- 3. Split: val 1800 balanced + train 8200 (no overlap) ---
def sample_balanced(df, n_per_class=450, seed=42):
    assert n_per_class % 3 == 0
    parts = []
    for label in LABEL_COLS:
        sub = df[df['label'] == label]
        if label == 'same_figure':
            parts.append(sub[sub['pair_type'] == 'TXT-IMG'].sample(n=n_per_class, random_state=seed))
        else:
            per_pair = n_per_class // 3
            for pt in ['TXT-IMG','IMG-IMG','TXT-TXT']:
                g = sub[sub['pair_type'] == pt]
                parts.append(g.sample(n=per_pair, random_state=seed))
    bal = pd.concat(parts).sample(frac=1, random_state=seed)
    return bal
val = sample_balanced(train, n_per_class=VAL_N_PER_CLASS, seed=42)
train_remain = train.drop(val.index)
assert len(train_remain) == len(train) - len(val)
assert set(train_remain.index).isdisjoint(set(val.index)), 'OVERLAP'
train_remain = train_remain.reset_index(drop=True)
print('val ' + str(val.shape))
print(val['label'].value_counts())
print(pd.crosstab(val['label'], val['pair_type']))
print('train_remain ' + str(train_remain.shape))


val (1800, 12)
label
unrelated_papers    450
related_papers      450
same_paper          450
same_figure         450
Name: count, dtype: int64
pair_type         IMG-IMG  TXT-IMG  TXT-TXT
label                                      
related_papers        150      150      150
same_figure             0      450        0
same_paper            150      150      150
unrelated_papers      150      150      150
train_remain (8200, 12)


In [6]:
# --- 4. Prompt + builders (same contract as SmolVLM baseline) ---
SYSTEM_PROMPT = '''You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.

Classes:
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned verbatim in the text, or the text reads like "Figure X shows..." matching the image.
- same_paper: Same study, different figures. Similar writing style, same instruments/datasets/authors hinted in text, or visual style (fonts, colors, layout) is consistent, but NOT a direct caption-figure match.
- related_papers: Different papers where one builds on the other. Overlapping methods, shared datasets, or a figure/caption that looks like a cited prior result, but style/authors differ.
- unrelated_papers: No clear link. Different topics, instruments, scales, or writing/visual style with no overlap.

Base your decision only on visual and textual content. Do not assume same_figure is impossible for any pair type - judge from alignment.
Output ONLY the lowercase label (e.g., related_papers), no explanation, no punctuation.
'''
def build_user_content(row):
    parts = []
    for col, name in [('obj_1','Object A'), ('obj_2','Object B')]:
        s = row[col]
        if row[col + '_is_img']:
            parts.append({'name': name, 'is_img': True, 'pil': convert_str_to_PIL(s)})
        else:
            parts.append({'name': name, 'is_img': False, 'text': str(s)[:CAP_CHARS]})
    return parts
def build_messages(row):
    parts = build_user_content(row)
    content = [{'type': 'text', 'text': SYSTEM_PROMPT + '\n'}]
    for p in parts:
        if p['is_img']:
            im = p['pil'].copy(); im.thumbnail((448, 448))
            content.append({'type': 'image', 'image': im})
        if p['is_img']:
            content.append({'type': 'text', 'text': '\n' + p['name'] + ': [Figure image]'})
        else:
            content.append({'type': 'text', 'text': '\n' + p['name'] + ' (caption): ' + p['text']})
    content.append({'type': 'text', 'text': '\nAnswer with one label:'})
    return [{'role': 'user', 'content': content}]
import re
label_pattern = re.compile('(same_figure|same_paper|related_papers|unrelated_papers)', re.IGNORECASE)
def parse_label(text):
    m = label_pattern.search(str(text).lower()); return m.group(1).lower() if m else 'unrelated_papers'
_demo = build_messages(train_remain.iloc[0])
print('demo parts: ' + str(len(_demo[0]['content'])))


demo parts: 5


In [7]:
# --- 5. Model: SmolVLM-500M bf16 + LoRA (2xT4, no quantization needed) ---
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as ModelClass; print('Using AutoModelForImageTextToText')
except:
    from transformers import AutoModel as ModelClass; print('Using AutoModel fallback')
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print('Processor: ' + type(processor).__name__)
_ip = processor.image_processor
if hasattr(_ip, 'do_image_splitting'):
    _ip.do_image_splitting = False
    print('image_splitting OFF -> single global tile per image (kills 13-tile blowup)')
print('vision caps: do_image_splitting=' + str(getattr(_ip, 'do_image_splitting', 'n/a')) + ' size=' + str(getattr(_ip, 'size', 'n/a')))
model = ModelClass.from_pretrained(MODEL_ID, device_map='auto', max_memory={0:'15GB',1:'15GB'}, torch_dtype=torch.float16, trust_remote_code=True)
print('Model bf16: ' + type(model).__name__)
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.config.use_cache = False
_tok = processor.tokenizer
if _tok.pad_token_id is None or _tok.pad_token_id >= len(_tok):
    _tok.pad_token = _tok.eos_token
    model.config.pad_token_id = _tok.pad_token_id
    print('pad_token repaired -> id ' + str(_tok.pad_token_id))
else:
    print('pad_token OK id=' + str(_tok.pad_token_id) + ' vocab=' + str(len(_tok)))
try:
    model.gradient_checkpointing_enable()
    print('gradient-checkpointing enabled')
except Exception as _e:
    print('ckpt enable skipped: ' + str(_e)[:200])
print('Has generate: ' + str(hasattr(model, 'generate')))


Using AutoModelForImageTextToText


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.02GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model bf16: Idefics3ForConditionalGeneration
trainable params: 19,136,512 || all params: 526,618,816 || trainable%: 3.6338
pad_token OK id=2 vocab=49280
gradient-checkpointing enabled
Has generate: True


In [8]:
# --- 6. Train/val lists + ANSWER-ONLY masked collator (prompt masked, label kept) ---
def row_to_messages_and_label(row):
    messages = build_messages(row)
    images = []
    for turn in messages:
        for part in turn['content']:
            if part.get('type') == 'image': images.append(part['image'])
    return {'messages': messages, 'images': images, 'label': row['label'], 'meta': (str(row['label']), str(row['pair_type']), str(row.get('id', '')))}
train_ds_list = [row_to_messages_and_label(train_remain.iloc[i]) for i in range(len(train_remain))]
val_ds_list = [row_to_messages_and_label(val.iloc[i]) for i in range(len(val))]
print('train ' + str(len(train_ds_list)) + ' val ' + str(len(val_ds_list)))
_ENCODE_FAILS = 0; _CACHE_USES = 0; _FULLMASK = 0  # training-health counters (nonzero = bug, not noise)
def collate_fn(features):
    prompts, fulls, img_lists = [], [], []
    for f in features:
        p = processor.apply_chat_template(f['messages'], tokenize=False, add_generation_prompt=False)
        prompts.append(p)
        fulls.append(p + '\n' + f['label'])
        img_lists.append(list(f['images']))
    def _encode(_prompts, _fulls, _imgs, _flat):
        if _flat:
            _imgarg = [im for _sub in _imgs for im in _sub] or None
        else:
            _imgarg = _imgs
        if _imgarg:
            _ef = processor(text=_fulls, images=_imgarg, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
            _ep = processor(text=_prompts, images=_imgarg, padding=False, truncation=True, max_length=MAX_SEQ_LEN, return_tensors=None)
        else:
            _ef = processor(text=_fulls, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
            _ep = processor(text=_prompts, padding=False, truncation=True, max_length=MAX_SEQ_LEN, return_tensors=None)
        return _ef, _ep
    enc_full, enc_prompt, _kept = None, None, None
    _all = list(range(len(fulls)))
    try:
        enc_full, enc_prompt = _encode(prompts, fulls, img_lists, len(fulls) == 1)
        _kept = _all
    except Exception as _e0:
        print('Batch encode failed, shrinking: ' + str(_e0)[:160])
        for _keep in range(len(fulls) - 1, 0, -1):
            _idx = sorted(range(len(fulls)), key=lambda i: len(fulls[i]))[:_keep]
            try:
                enc_full, enc_prompt = _encode([prompts[i] for i in _idx], [fulls[i] for i in _idx], [img_lists[i] for i in _idx], len(_idx) == 1)
                _kept = _idx
                print('WARN: dropped ' + str(len(fulls) - _keep) + ' sample(s) from micro-batch')
                break
            except Exception as _e1:
                print('Encode failed keep=' + str(_keep) + ' ' + str(_e1)[:160])
        if enc_full is None:
            for i in _all:
                try:
                    enc_full, enc_prompt = _encode([prompts[i]], [fulls[i]], [img_lists[i]], True)
                    _kept = [i]
                    print('WARN: flat singleton worked for sample ' + str(i))
                    break
                except Exception as _e2:
                    globals()['_ENCODE_FAILS'] = globals().get('_ENCODE_FAILS', 0) + 1
                    _m = features[i].get('meta', ('?', '?', '?'))
                    print('singleton failed i=' + str(i) + ' meta=' + str(_m) + ' chars=' + str(len(fulls[i])) + ' imgs=' + str(len(img_lists[i])) + ' err=' + str(_e2)[:160] + ' (fail #' + str(globals()['_ENCODE_FAILS']) + ')')
        if enc_full is None:
            _gc = globals().get('_GOOD_CACHE')
            if _gc is not None:
                globals()['_CACHE_USES'] = globals().get('_CACHE_USES', 0) + 1
                if globals()['_CACHE_USES'] % 10 == 1:
                    print('WARN: training on cached sample instead (use #' + str(globals()['_CACHE_USES']) + ') - INVESTIGATE, do not ignore')
                enc_full, enc_prompt = _encode(_gc[0], _gc[1], _gc[2], True)
            else:
                raise RuntimeError('collate encode failed for all fallbacks')
    if globals().get('_GOOD_CACHE') is None and _kept is not None:
        globals()['_GOOD_CACHE'] = ([prompts[i] for i in _kept], [fulls[i] for i in _kept], [img_lists[i] for i in _kept])
    input_ids = enc_full['input_ids']
    labels = input_ids.clone()
    prompt_lens = [len(ids) for ids in enc_prompt['input_ids']]
    for i, pl in enumerate(prompt_lens):
        pl = min(pl, int(input_ids.shape[1]))
        labels[i, :pl] = -100
    labels[enc_full['attention_mask'] == 0] = -100
    if int((labels != -100).sum()) == 0:
        globals()['_FULLMASK'] = globals().get('_FULLMASK', 0) + 1
        print('WARN: batch fully masked, supervising last tokens (event #' + str(globals()['_FULLMASK']) + ')')
        for i in range(input_ids.shape[0]):
            L = int(enc_full['attention_mask'][i].sum())
            labels[i, L-1] = input_ids[i, L-1]
    enc_full['labels'] = labels
    if 'pixel_values' in enc_full and torch.is_floating_point(enc_full['pixel_values']):
        enc_full['pixel_values'] = enc_full['pixel_values'].to(torch.float16)
    return enc_full
print('collator ready (answer-only loss)')
model.train()
try:
    _dev = model.device
except Exception:
    _dev = next(model.parameters()).device
_batch = collate_fn(train_ds_list[:BATCH_SIZE])
print({k: (tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__) for k, v in _batch.items()})
print('supervised answer tokens: ' + str(int((_batch['labels'] != -100).sum())) + ' / total ' + str(int(_batch['attention_mask'].sum())))
_tiles = int(_batch['pixel_values'].shape[1]) if 'pixel_values' in _batch else 0
print('image tiles per sample: ' + str(_tiles))
assert int((_batch['labels'] != -100).sum()) >= 1, 'REFUSING TO LAUNCH: 0 supervised tokens - fix masking/truncation first'
assert _tiles <= 4, 'REFUSING TO LAUNCH: ' + str(_tiles) + ' tiles - check do_image_splitting/thumbnail'
for _pt in ['TXT-IMG', 'IMG-IMG', 'TXT-TXT']:
    _si = next((i for i, f in enumerate(train_ds_list) if f['meta'][1] == _pt), None)
    assert _si is not None, 'no ' + _pt + ' sample found'
    _sb = collate_fn([train_ds_list[_si]])
    _s = int((_sb['labels'] != -100).sum())
    print('smoke ' + _pt + ' supervised=' + str(_s) + ' tiles=' + str(int(_sb['pixel_values'].shape[1]) if 'pixel_values' in _sb else 0))
    assert _s >= 1, 'REFUSING TO LAUNCH: ' + _pt + ' sample supervises 0 tokens'
    del _sb
print('SMOKE GATE PASSED: all pair types encode with real supervision')
with torch.no_grad():
    _out = model(**{k: (v.to(_dev) if hasattr(v, 'to') else v) for k, v in _batch.items()})
print('Forward OK loss: ' + str(round(float(_out.loss), 4)))
del _batch, _out; torch.cuda.empty_cache()


train 8200 val 1800
collator ready (answer-only loss)
{'input_ids': (1, 470), 'attention_mask': (1, 470), 'pixel_values': (1, 1, 3, 512, 512), 'pixel_attention_mask': (1, 1, 512, 512), 'labels': (1, 470)}
supervised answer tokens: 4 / total 470
image tiles per sample: 1
smoke TXT-IMG supervised=4 tiles=1
smoke IMG-IMG supervised=4 tiles=2
smoke TXT-TXT supervised=4 tiles=0
SMOKE GATE PASSED: all pair types encode with real supervision
Forward OK loss: 4.4644


In [9]:
# --- 7. Train: LoRA + early stopping + HF intermediate pushes ---
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback
class HubPushCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        try:
            if int(state.global_step) % 400 != 0: return
            if not HF_TOKEN or not HF_REPO_ID: return
            from huggingface_hub import HfApi; _api = HfApi()
            _api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
            _ckpt = CKPT_DIR + '/checkpoint-' + str(int(state.global_step))
            _api.upload_folder(repo_id=HF_REPO_ID, folder_path=_ckpt, path_in_repo='checkpoints/checkpoint-' + str(int(state.global_step)), commit_message='ckpt ' + str(int(state.global_step)))
            print('Pushed ckpt ' + str(int(state.global_step)))
        except Exception as _e: print('Push failed: ' + str(_e)[:200])
training_args = TrainingArguments(output_dir=CKPT_DIR, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM, max_steps=MAX_STEPS, learning_rate=LR, lr_scheduler_type='cosine', warmup_steps=256, fp16=True, gradient_checkpointing=True, logging_steps=10, save_steps=SAVE_STEPS, save_strategy='steps', eval_strategy='steps', eval_steps=EVAL_STEPS, logging_strategy='steps', save_total_limit=3, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, push_to_hub=False, report_to='none', remove_unused_columns=False)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds_list, eval_dataset=val_ds_list, data_collator=collate_fn, compute_metrics=None, callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD), HubPushCallback()])
print('eff batch ' + str(BATCH_SIZE*GRAD_ACCUM) + ' steps/ep ' + str(len(train_ds_list)//(BATCH_SIZE*GRAD_ACCUM)))
TRAIN_T0 = time.time()
from pathlib import Path as _P
_ckpts = sorted(_P(training_args.output_dir).glob('checkpoint-*'), key=lambda q: q.stat().st_mtime)
if not _ckpts and HF_TOKEN:
    try:
        from huggingface_hub import HfApi, snapshot_download
        _rf = HfApi().list_repo_files(repo_id=HF_REPO_ID, repo_type='model')
        _steps = sorted([int(p.split('checkpoint-')[-1].split('/')[0]) for p in _rf if p.startswith('checkpoints/checkpoint-') and p.split('checkpoint-')[-1].split('/')[0].isdigit()])
        if _steps:
            _latest = _steps[-1]
            _dl = snapshot_download(repo_id=HF_REPO_ID, allow_patterns=['checkpoints/checkpoint-' + str(_latest) + '/*'], repo_type='model')
            import shutil as _sh; _loc = CKPT_DIR + '/checkpoint-' + str(_latest); _sh.rmtree(_loc, ignore_errors=True); _sh.copytree(_dl + '/checkpoints/checkpoint-' + str(_latest), _loc)
            _ckpts = sorted(_P(training_args.output_dir).glob('checkpoint-*'), key=lambda q: q.stat().st_mtime)
            print('Resumed from HF checkpoints/checkpoint-' + str(_latest))
        else: print('No HF checkpoints yet - training from scratch')
    except Exception as _e: print('HF resume skipped: ' + str(_e)[:200])
os.makedirs(EVAL_DIR, exist_ok=True)
open(os.path.join(EVAL_DIR, 'run_config.json'), 'w').write(json.dumps({'model': MODEL_ID, 'lora_r': LORA_R, 'lr': LR, 'batch': BATCH_SIZE, 'grad_accum': GRAD_ACCUM, 'max_steps': MAX_STEPS, 'repo': HF_REPO_ID}))
np.save(os.path.join(EVAL_DIR, 'val_ids.npy'), val['id'].values if 'id' in val.columns else np.arange(len(val)))
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi as _Hf; _h = _Hf(); _h.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
        _h.upload_file(path_or_fileobj=os.path.join(EVAL_DIR, 'run_config.json'), path_in_repo='run_config.json', repo_id=HF_REPO_ID)
        _h.upload_file(path_or_fileobj=os.path.join(EVAL_DIR, 'val_ids.npy'), path_in_repo='val_ids.npy', repo_id=HF_REPO_ID)
    except Exception as _e: print('config push skipped: ' + str(_e)[:200])
trainer.train(resume_from_checkpoint=str(_ckpts[-1]) if _ckpts else None)
print('best ' + str(trainer.state.best_metric) + ' at ' + str(trainer.state.best_model_checkpoint))
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi; api = HfApi()
        api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
        api.upload_folder(repo_id=HF_REPO_ID, folder_path=FINAL_DIR, path_in_repo='adapter_best', commit_message='best eval_loss ' + str(round(float(trainer.state.best_metric), 4)))
        print('Pushed adapter_best')
    except Exception as e: print('Push best failed ' + str(e)[:300])
else:
    print('No HF_TOKEN — adapter kept local: ./smolvlm_500m_lora_final')


eff batch 16 steps/ep 512
No HF checkpoints yet - training from scratch


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Step,Training Loss,Validation Loss
200,0.290182,0.349105
400,0.296175,0.302887
600,0.289620,0.369982
800,0.280653,0.319209
1000,0.279392,0.329669


Pushed ckpt 400
Pushed ckpt 800
best 0.302887499332428 at /kaggle/working/smolvlm_500m_lora_out/checkpoint-400
Pushed adapter_best


In [10]:
# --- 8. Generation eval on val 1800 + metrics + plots + push ---
OUT_DIR = EVAL_DIR
os.makedirs(OUT_DIR, exist_ok=True)
JSONL = os.path.join(OUT_DIR, 'raw_outputs.jsonl')
open(JSONL, 'w').close()
model.eval()
try:
    _DEV = model.device
except Exception:
    _DEV = next(model.parameters()).device
y_true, y_pred = [], []
n_oom, n_parse_fb = 0, 0
_jf = open(JSONL, 'a')
t0 = time.time()
for idx in tqdm(range(len(val))):
    if time.time() - TRAIN_T0 > TIME_BUDGET:
        print('TIME GUARD - scoring prefix'); break
    row = val.iloc[idx]
    try:
        msgs = build_messages(row)
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        fimgs = []
        for turn in msgs:
            for part in turn['content']:
                if part.get('type') == 'image': fimgs.append(part['image'])
        fimgs = fimgs if fimgs else None
        inputs = processor(text=[text], images=fimgs, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
    except Exception as _e:
        print('Eval encode failed idx ' + str(idx) + ', text fallback: ' + str(_e)[:200])
        inputs = processor(text=['Object A: undecodable. Object B: undecodable. Answer with one label:'], padding=True, return_tensors='pt')
    inputs = {k: (v.to(_DEV) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    if 'pixel_values' in inputs: inputs['pixel_values'] = inputs['pixel_values'].to(torch.float16)
    L = inputs['input_ids'].shape[1]
    try:
        with torch.no_grad(): out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    except RuntimeError as e:
        if 'out of memory' not in str(e).lower(): raise
        torch.cuda.empty_cache(); gc.collect()
        decoded = ''; n_oom += 1; pred = 'unrelated_papers'
    else:
        decoded = processor.batch_decode(out[:, L:], skip_special_tokens=True)[0]; pred = parse_label(decoded)
    true = row['label']
    if pred == 'unrelated_papers' and 'unrelated_papers' not in str(decoded).lower(): n_parse_fb += 1
    y_true.append(true); y_pred.append(pred)
    rec = {'idx': int(idx), 'id': str(row.get('id', idx)), 'true_label': str(true), 'pair_type': str(row['pair_type']), 'pred': pred, 'raw': decoded, 'thinking': '', 'decoded': pred}
    _jf.write(json.dumps(rec) + '\n')
    if len(y_true) % 200 == 0:
        _jf.flush()
        print('ckpt ' + str(len(y_true)) + ' elapsed ' + str(round((time.time()-t0)/60, 1)) + 'min', flush=True)
    del inputs
    if idx % 50 == 0: torch.cuda.empty_cache()
_jf.close()
y_true = np.array(y_true); y_pred = np.array(y_pred)
acc = accuracy_score(y_true, y_pred); macro = f1_score(y_true, y_pred, average='macro')
per_class = f1_score(y_true, y_pred, average=None, labels=LABEL_COLS)
print('Overall acc=' + str(round(float(acc), 4)) + ' macro-F1=' + str(round(float(macro), 4)) + ' on ' + str(len(y_true)) + '/' + str(len(val)))
print(dict(zip(LABEL_COLS, per_class.round(4))))
print(classification_report(y_true, y_pred, labels=LABEL_COLS, digits=4))
cm = confusion_matrix(y_true, y_pred, labels=LABEL_COLS)
print(pd.DataFrame(cm, index=LABEL_COLS, columns=LABEL_COLS))
for pt in ['TXT-IMG','IMG-IMG','TXT-TXT']:
    m = (val.iloc[:len(y_true)]['pair_type'] == pt).values
    if m.sum() == 0: continue
    print(pt + ' n=' + str(int(m.sum())) + ' macro-F1=' + str(round(float(f1_score(y_true[m], y_pred[m], average='macro'))), 4)))
plt.figure(figsize=(7,5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABEL_COLS, yticklabels=LABEL_COLS)
plt.title('Confusion val - smolvlm-500m-lora macro-F1 ' + str(round(float(macro), 3))); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion.png'), dpi=150); plt.show()
plt.figure(figsize=(6,3)); plt.bar(LABEL_COLS, per_class); plt.title('Per-class F1'); plt.ylim(0,1); plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'per_class_f1.png')); plt.show()
metrics = {'model': MODEL_ID, 'overall': {'acc': float(acc), 'macro_f1': float(macro), 'per_class': dict(zip(LABEL_COLS, per_class.tolist()))}, 'n_scored': int(len(y_true)), 'n_total': int(len(val)), 'n_oom': int(n_oom), 'n_parse_fb': int(n_parse_fb)}
open(os.path.join(OUT_DIR, 'metrics.json'), 'w').write(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
import shutil; shutil.copy(JSONL, os.path.join(OUT_DIR, 'raw_outputs_final.jsonl'))
np.save(os.path.join(OUT_DIR, 'preds_val_final.npy'), np.array(y_pred))
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi; api2 = HfApi()
        api2.upload_folder(repo_id=HF_REPO_ID, folder_path=OUT_DIR, path_in_repo='eval', commit_message='val macro-F1 ' + str(round(float(macro), 4)))
        print('Pushed eval/')
    except Exception as e: print('eval push failed ' + str(e)[:300])


SyntaxError: unmatched ')' (1418986024.py, line 64)

In [ ]:
# --- 9. Summary ---
print('Model: ' + MODEL_ID + ' LoRA r' + str(LORA_R))
print('Train ' + str(len(train_remain)) + ' / val scored ' + str(len(y_true)) + '/' + str(len(val)))
print('val acc=' + str(round(float(acc), 4)) + ' macro-F1=' + str(round(float(macro), 4)))
print('HF: ' + str(HF_REPO_ID) + ' (adapter_best + checkpoints/* + eval/)')
print('Next: test.csv -> submission.csv with adapter (PeftModel on 2xT4)')
